In [3]:
## Import necessary packages

%matplotlib osx 
# On Macs use osx
# For Windows use qt

import numpy as np
from numpy.random import rand
from landscapeWithOcean import LandscapeWithOcean # Import methods from inside file landscapeWithOcean.py

from mpl_toolkits.mplot3d import Axes3D
import matplotlib.pyplot as plt
from matplotlib import cm

In [4]:
def AddHill(Z,NX,NY,xx,yy,r,h):

    for x in range(NX):
        for y in range(NY):
            dx = np.mod(x-xx+NX/2,NX)-NX/2; # difference i-i0 but apply p.b.c. 
            dy = np.mod(y-yy+NY/2,NY)-NY/2;
            dr = np.sqrt(dx**2+dy**2);
            if (dr<r):
                Z[x,y] += h * (np.cos(dr/r*np.pi/2.0))**2;

    return Z

In [5]:
### Define simulation grid and initial conditions

NX = 70*2 #number of rows
NY = 70*2 #number of columns

d  = 5 # grid spacing in meters
dx = d # keep dx=dy for simplicity
dy = d

LX=NX*dx
LY=NY*dy

# small random features in topography to begin erosion
Z = rand(NX,NY)*0.1
for i in range(5):
    xx = rand()*NX
    yy = rand()*NY
    r  = (0.1+rand())*NX
    h  = (0.1+rand())*5
    Z = AddHill(Z,NX,NY,xx,yy,r,h)

for i in range(5):
    xx = rand()*NX
    yy = rand()*NY
    r  = (0.1+rand())*NX/2
    h  = (0.5+rand())*10
    Z = AddHill(Z,NX,NY,xx,yy,r,h)

x = np.arange(NX)
y = np.arange(NY)
X,Y = np.meshgrid(y,x) #strange that y goes first !!!
ZMaxOrg = np.max(Z)
print(ZMaxOrg)

22.968427675317365


In [6]:
### Physical Parameters
K = 1.0e-6 # meters^(1-2m)/yr

D = 0.005 # m^2/yr

# uplift rate
# uplift = 0.03 / 600.
uplift = 0.0

In [7]:
### Model parameters

# Time step
dt = d**2 / D / 8. 
#dt = d**2 / D /16. #extra small steps 
print(' dt[years] = ',dt)

#Area exponent A^m, default m=1
m=1

#gradient exponent g^n, default n=1
n=1

#erosion threshold 
theta_c = 10 

 dt[years] =  625.0


In [8]:
# Total simulation time
T = 1000.0 * 625.0

# total number of iterations
n_iter = int(np.round(T/dt))
print('Number of interation: ',n_iter)

Number of interation:  1000


In [9]:
# Initialize landscape 
ls = LandscapeWithOcean(NX,NY)

oceanLevelParameter=0.1  # what does this parameter do?
ls.ComputeOceanVolumeFromOceanLevelParameter(Z,NX,NY,oceanLevelParameter)

ls.pool_check(Z,NX,NY)
ls.A = np.zeros((NX,NY))

Minimum elevation           3.872361310605325
Maximum elevation           22.968427675317365
Beach level                 5.781967947076529
Ocean volume                2269.7021608255036
Percentage of ocean surface 18.23469387755102


In [10]:
# Set-up figure
def init_figure():
    fig = plt.figure(figsize=(12.,6.))
    plt.show()
    return fig

def update_figure():
        plt.clf()
        ax1 = fig.add_subplot(121,projection='3d')

        # use equal x-y aspect with an explicit vertical exageration
        vert_exag = 4.
        ax1.set_xlim3d(0,max(NX,NY))
        ax1.set_ylim3d(0,max(NX,NY))
        ax1.set_zlim3d(0,ZMaxOrg)

        ax1.set_title('Surface Relief')

#        surf = ax1.plot_surface(X,Y,Z, cmap = cm.terrain, rstride=1, cstride=1,
#                antialiased=False,linewidth=0)

        ZPlot = np.copy(Z)
        ZPlot[ZPlot<ls.ZBeachLevel] = ls.ZBeachLevel 
        ZPlot -= ls.ZBeachLevel
        ax1.plot_surface(X,Y,ZPlot, cmap = cm.terrain, rstride=1, cstride=1,
                antialiased=False,linewidth=0)

        ax2 = fig.add_subplot(122,aspect='equal')
        ax2.set_title('Elevation')

        #im = ax2.pcolor(Z,cmap=cm.terrain)
        im = ax2.pcolor(ZPlot,cmap=cm.coolwarm)
        cs = ax2.contour(ZPlot,6,colors='k')

        # Add a color bar which maps values to colors.
        cbar = fig.colorbar(im, shrink=0.5, aspect=5)
        # Add the contour line levels to the colorbar
        cbar.add_lines(cs)

        #plt.show()
        plt.draw()
        plt.pause(0.0001)

In [28]:
# Set up figure
fig = init_figure()
update_figure()
Znew = np.copy(Z)

for it in range(1,n_iter+1):
    
    ls.calculate_collection_area(Z,NX,NY)
    ls.A *= dx*dy
    
    for i in range(NX):
        iL = np.mod(i-1,NX) # normally i-1 but observe p.b.c.
        iR = np.mod(i+1,NX) # normally i+1 but observe p.b.c.

        for j in range(NY):
            jD = np.mod(j-1,NY) # normally j-1 but observe p.b.c.
            jU = np.mod(j+1,NY) # normally j+1 but observe p.b.c.
  
            if ls.ocean[i,j]>0:
                Psi_z  = 0;
                Phi_z  = 0;
            else:
                if ls.drain[i,j]>0: #this cell is a drain
                    s1 = (Z[iR,j] - Z[iL,j])/ (2 * dx)
                    s2 = (Z[i, jU] - Z[i, jD]) / (2 * dy)
                    s3 = (Z[iR,jD] - Z[iL, jU]) / (2 * np.sqrt(dx**2 + dy**2))
                    s4 = (Z[iR,jU] - Z[iL, jD])  / (2 * np.sqrt(dx**2 + dy**2))
                    gradient =  np.sqrt(1/2) * np.sqrt(s1**2 + s2**2 + s3**2 + s4**2)
                    Psi_z = K * ( ls.A[i,j]**m * gradient**n - theta_c)            

                elif ls.drainage[i,j]>0: #this cell is a drainage point (it drains a pool)
                
                    if (Z[i,j]>=Z[iR,j]) and ls.pool[iR,j]!=ls.drainage[i,j]: 
                        gradient = (Z[i,j]-Z[iR,j])/dx #pool is on my left, I drain to the right, use this gradiant
                    elif (Z[i,j]>=Z[iL,j]) and ls.pool[iL,j]!=ls.drainage[i,j]:
                        gradient = (Z[i,j]-Z[iL,j])/dx
                    elif (Z[i,j]>=Z[i,jU]) and ls.pool[i,jU]!=ls.drainage[i,j]:
                        gradient = (Z[i,j]-Z[i,jU])/dy
                    elif (Z[i,j]>=Z[i,jD]) and ls.pool[i,jD]!=ls.drainage[i,j]:
                        gradient = (Z[i,j]-Z[i,jD])/dy
                    else:
                        gradient = 0.02 # ??? This does happen (maybe when two pools merge)
                    Psi_z = K * ( ls.A[i,j]**m * gradient**n - theta_c)
                
                else: #this cell is a pool, assume it has some mass diffusion but no erosion!
                    Psi_z = 0.
                
                if (Psi_z<0):
                    Psi_z = 0. 

                # diffusion term
                Phi_z = D * (((Z[iR, j] - 2*Z[i,j] + Z[iL,j]) / dx**2) + ((Z[i,jU] - 2*Z[i,j] + Z[i,jD]) / dy**2))
           
            Znew[i,j] = Z[i,j] + (Phi_z - Psi_z + uplift )*dt  

            dZdt= (Znew[i,j] - Z[i,j]) / dt
            CFL = abs(dZdt) * dt / min(dx,dy)
            if (CFL>1.0):
                print('\nWarning: Time step of',dt,'is probably too large. Safer would be:',dt/CFL)
            
            if (Znew[i,j]<0.):
                Znew[i,j] = 0. # yes, this does happen at the boundary when kept at zero
    
    #Znew[0,:] = 0.0 # resets front boundary to 0
    Z = np.copy(Znew)
    
    ls.pool_check(Z,NX,NY)

    if (np.mod(it,10)==0): 
        print(it,end='')
        update_figure()
        print(' Ocean level=',ls.ZBeachLevel,' Ocean surface fraction=',100*ls.AOcean/(NX*NY));
    else:
        print('.',end='')

update_figure()
print(' Simulation finished.')


......
.
.
.10 Ocean level= 4.385268203931431  Ocean surface fraction= 7.224489795918367


.
.
.
..

.
.
..

20 Ocean level= 4.349873397932961  Ocean surface fraction= 7.142857142857143

.
.
..

.
.
.
..

30 Ocean level= 4.316682096031669  Ocean surface fraction= 7.061224489795919

.
.
.
.
.
.
...
40 Ocean level= 4.28675088612178  Ocean surface fraction= 7.3061224489795915
.........50 Ocean level= 4.277587058794039  Ocean surface fraction= 7.545918367346939
.........60 Ocean level= 4.271187336572992  Ocean surface fraction= 7.821428571428571
.........70 Ocean level= 4.2680345024619735  Ocean surface fraction= 8.051020408163266
.........80 Ocean level= 4.263789870367295  Ocean surface fraction= 8.11734693877551
.........90 Ocean level= 4.262469508910126  Ocean surface fraction= 8.204081632653061
.........100 Ocean level= 4.2604522762724875  Ocean surface fraction= 8.433673469387756
.........110 Ocean level= 4.259122264314476  Ocean surface fraction= 8.576530612244898
.........120 Ocean 

### Write Up:

Intial conditions: The initial conditions represent hills. They are randomly generated. There is an initial default terrain generated with the Z=rand(NX,NY)*0.1, and then it is altered even further with the AddHill function. The AddHill function is called on random areas of the terrain. It adds some height h within some radius r. The height of the hill tapers off to the sides with the proportionality of a cosine wave (from peak to node). 

The oceanlevelparameter variable helps determine what the starting height of the water is in the simulation. It places the water level just above the ground level, in order to give an upper bound for the water depth. It is used to calculate the "beach level" which is necessary for finding the total ocean volume. 

Over time the amount of land above water decreases. It erodes beneath the ocean level, increasing the surface area of the water. 

In [2]:
def create_custom_landscape(NX, NY):
    # Initialize a zero-elevation grid of dimensions NX by NY
    Z = np.zeros((NX, NY))
    
    # Generate 2D index coordinate arrays for deterministic analytical functions
    x_idx, y_idx = np.meshgrid(np.arange(NX), np.arange(NY), indexing='ij')
    
    # 1. Add a long-wavelength sinusoidal swell to represent a continental shelf and adjacent basin
    Z += 1.8 * (np.sin(2.0 * np.pi * x_idx / NX) + np.cos(2.0 * np.pi * y_idx / NY))
    
    # Add high-frequency deterministic interference ripples to seed channelization without random numbers
    Z += 0.25 * (np.sin(2.0 * np.pi * x_idx / 6.0) * np.cos(2.0 * np.pi * y_idx / 8.0) + 
                 0.5 * np.cos(2.0 * np.pi * (x_idx + y_idx) / 5.0))
    
    # 2. Construct the central Somma-type volcanic caldera at (0.42*NX, 0.42*NY)
    cx, cy = NX * 0.42, NY * 0.42
    # Add the broad outer volcanic shield edifice (positive height h = 22.0 m)
    Z = AddHill(Z, NX, NY, cx, cy, r=NX * 0.38, h=22.0)
    # Subtract a central circular depression to form the collapsed caldera crater (negative height h = -12.0 m)
    Z = AddHill(Z, NX, NY, cx, cy, r=NX * 0.17, h=-12.0)
    # Add a smaller central peak inside the crater to represent a resurgent lava dome (h = 7.5 m)
    Z = AddHill(Z, NX, NY, cx + 2, cy - 2, r=NX * 0.07, h=7.5)
    
    # 3. Excavate a saddle notch on the SE caldera rim to serve as a crater lake spillway (h = -4.0 m)
    Z = AddHill(Z, NX, NY, cx + NX * 0.14, cy + NY * 0.10, r=NX * 0.08, h=-4.0)
    
    # 4. Superpose four overlapping domes in the NE quadrant to form an arcuate tectonic ridge / island arc
    ridge_points = [
        (NX * 0.80, NY * 0.25, NX * 0.20, 16.0),
        (NX * 0.84, NY * 0.48, NX * 0.18, 18.5),
        (NX * 0.76, NY * 0.74, NX * 0.22, 20.0),
        (NX * 0.58, NY * 0.84, NX * 0.17, 15.0),
    ]
    for xx, yy, r, h in ridge_points:
        Z = AddHill(Z, NX, NY, xx, yy, r, h)
        
    # 5. Add an isolated coastal headland / sea-stack in the NW quadrant
    Z = AddHill(Z, NX, NY, NX * 0.18, NY * 0.82, r=NX * 0.18, h=12.0)
    
    # Shift the entire grid vertically so the minimum elevation is strictly positive (0.1 m)
    Z = Z - np.min(Z) + 0.1
    return Z